In [1]:
import os
import gymnasium
import highway_env
import warnings
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback, CallbackList

env_name = 'highway-v0'
root = f'{env_name}-PPO'

warnings.filterwarnings("ignore", category=DeprecationWarning)

# ── Configuración ────────────────────────────────────────────────────────────
config = {
    "observation": {
        "type": "Kinematics",
        "vehicles_count": 15,
        "features": ["presence", "x", "y", "vx", "vy"],
        "features_range": {
            "x": [-100, 100],
            "y": [-100, 100],
            "vx": [-30, 30],
            "vy": [-30, 30],
        },
        "absolute": False,
        "order": "sorted",
        "normalize": True,
    },

    "action": {
        "type": "DiscreteMetaAction",
    },

    "lanes_count": 4,
    "vehicles_count": 50,
    "vehicles_density": 1.5,
    "duration": 60,
    "initial_lane_id": None,

    "simulation_frequency": 15,
    "policy_frequency": 5,

    "collision_reward": -4.0,
    "high_speed_reward": 1.0,
    "lane_change_reward": 0.05,
    "right_lane_reward": 0.0,
    "reward_speed_range": [25, 35],
    "normalize_reward": True,

    "offroad_terminal": True,
    "controlled_vehicles": 1,
    "manual_control": False,
}

log_dir = f"{root}/logs/"
os.makedirs(log_dir, exist_ok=True)

# Ruta del último modelo guardado para reanudación rápida
checkpoint_path = f"{log_dir}/latest_checkpoint.zip"

# ── Entornos ─────────────────────────────────────────────────────────────────
def make_env():
    env = gymnasium.make(env_name)
    env.unwrapped.configure(config)
    env.reset()
    env = Monitor(env, log_dir)
    return env

env      = DummyVecEnv([make_env])
eval_env = DummyVecEnv([make_env])

# ── Callbacks ─────────────────────────────────────────────────────────────────
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=log_dir,
    log_path=log_dir,
    eval_freq=20_000,
    n_eval_episodes=10,
    deterministic=True,
    verbose=0,
)

checkpoint_callback = CheckpointCallback(
    save_freq=20_000,
    save_path=log_dir,
    name_prefix="ppo_highway_step",
    save_replay_buffer=False,
)

callbacks = CallbackList([eval_callback, checkpoint_callback])

# ── Modelo (Carga o Creación) ──────────────────────────────────────────────────
TOTAL_TIMESTEPS = 200_000

if os.path.exists(checkpoint_path):
    print(f"¡Checkpoint detectado en {checkpoint_path}! Reanudando entrenamiento...")
    model = PPO.load(checkpoint_path)
    model.set_env(env)
else:
    print("No se encontró checkpoint previo. Creando modelo desde cero...")
    model = PPO(
        "MlpPolicy",
        env,
        verbose=1,
        n_steps=2048,
        batch_size=64,
        gae_lambda=0.95,
        gamma=0.99,
        n_epochs=10,
        learning_rate=3e-4,
        clip_range=0.2,
        ent_coef=0.03,
        policy_kwargs=dict(
            net_arch=[256, 256]
        ),
    )

# ── Entrenamiento ─────────────────────────────────────────────────────────────
try:
    print("Entrenando al agente...")
    model.learn(
        total_timesteps=TOTAL_TIMESTEPS, 
        callback=callbacks, 
        reset_num_timesteps=False
    )
except KeyboardInterrupt:
    print("\nEntrenamiento interrupted por el usuario (Ctrl+C). Guardando estado actual...")

finally:
    print("Guardando checkpoint de seguridad...")
    model.save(checkpoint_path)
    model.save(f"{log_dir}/last_model")
    
    env.close()
    eval_env.close()
    print("Entornos cerrados correctamente. ¡Listo para continuar en otro momento!")

<frozen importlib._bootstrap>:488: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.


No se encontró checkpoint previo. Creando modelo desde cero...
Using cpu device
Entrenando al agente...
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 22.3     |
|    ep_rew_mean     | 17.5     |
| time/              |          |
|    fps             | 8        |
|    iterations      | 1        |
|    time_elapsed    | 230      |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 25.7        |
|    ep_rew_mean          | 20.2        |
| time/                   |             |
|    fps                  | 9           |
|    iterations           | 2           |
|    time_elapsed         | 443         |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.011300886 |
|    clip_fraction        | 0.122       |
|    clip_range           | 0.2         |
|    entro